#### Imports and Load Data

Event definition (verified against the data, see `config.py`): a vehicle theft is a **crime class 510**
report, counted once per DR number. Class 521 rows are recoveries of stolen vehicles (they usually share the
DR of the 510 report) and class 997 rows are vehicles involved in traffic reports; neither is a theft event.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
import config
config.apply_plot_style()

In [ ]:
records = config.load_all_records()   # one row per (DR_NBR, crime class)
thefts  = config.load_theft_data()    # one row per 510 report, 2000-2022
rows_per_code = records.attrs["rows_per_code"]

print(f"Raw rows in preprocessed file      : {sum(rows_per_code.values()):,}")
print(f"Unique (DR, crime class) records    : {len(records):,}")
print(f"510 rows / unique 510 reports       : {rows_per_code['510']:,} / {records['code'].eq('510').sum():,}")
print(f"510 reports with occurrence 2000-22 : {len(thefts):,}")

#### Table 1 - Dataset composition

In [ ]:
years = config.STUDY_YEARS
rec = records[records["year"].between(years[0], years[-1])]
seg_counts = rec["segment"].value_counts()

n = len(thefts)
src = thefts["coord_source"].value_counts()
rows = [("Raw rows (all crime classes)", sum(rows_per_code.values()), ""),
        ("Unique (report, crime class) records, 2000-2022", len(rec), "")]
for seg in config.SEGMENT_CODES:
    rows.append((f"  {seg}", int(seg_counts.get(seg, 0)), f"{seg_counts.get(seg, 0)/len(rec)*100:.1f}% of records"))
rows += [
    ("Vehicle theft reports analysed (510, one per DR)", n, ""),
    ("  recovered (a 521 record shares the DR)", int(thefts["recovered"].sum()), f"{thefts['recovered'].mean()*100:.1f}%"),
    ("  occurrence time unknown (placeholder / not recorded)", int(thefts["time_unknown"].sum()), f"{thefts['time_unknown'].mean()*100:.1f}%"),
]
for s, label in [("stored", "Coordinates: stored (LAPD)"), ("census", "Coordinates: Census geocoder"),
                 ("nominatim", "Coordinates: Nominatim"), ("rejected_outside_city", "Coordinates: rejected (outside City of LA)"),
                 ("unresolved", "Coordinates: unresolved")]:
    rows.append((label, int(src.get(s, 0)), f"{src.get(s, 0)/n*100:.1f}%"))
with_xy = thefts["lat_final"].notna().sum()
rows.append(("Total with coordinates inside the city", int(with_xy), f"{with_xy/n*100:.1f}%"))

table1 = pd.DataFrame(rows, columns=["Category", "Count", "Note"])
table1.to_csv(os.path.join(config.TABLES_DIR, "table1_dataset_composition.csv"), index=False)
print(table1.to_string(index=False))

#### Data Segments Stacked Bar Chart

In [ ]:
seg_year = (rec.groupby(["year", "segment"]).size().unstack(fill_value=0)
            .reindex(index=years, columns=list(config.SEGMENT_CODES), fill_value=0))
colors = ["#b2182b", "#ef8a62", "#fddbc7", "#2166ac", "#67a9cf", "#d1e5f0"]

fig, ax = plt.subplots(figsize=config.FIGURE_SIZE_REGULAR)
bottom = np.zeros(len(years))
for seg, col in zip(seg_year.columns, colors):
    ax.bar(years, seg_year[seg], bottom=bottom, width=0.75, color=col, label=seg, zorder=2)
    bottom += seg_year[seg].values
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
ax.set_xlabel("Year", fontsize=18)
ax.set_ylabel("Records (unique report x class)", fontsize=14)
ax.set_title("Vehicle-Related LAPD Records by Crime Class", pad=12, fontsize=18)
ax.set_xticks(years)
ax.set_xticklabels(years, rotation=45, ha="right", fontsize=12)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=3, fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "fig1_data_stacked_bar.png"), dpi=config.FIGURE_DPI, bbox_inches="tight")
plt.show()

print(seg_year.to_string())

#### Coordinate Coverage by Year (regime check)

In 2001 and 2018-2022 the LAPD files carry (almost) no stored coordinates, so locations in those years come
from address geocoding. Any spatial pattern that changes exactly at these years must be checked against this.

In [ ]:
coord_source_order = ["stored", "census", "nominatim", "rejected_outside_city", "unresolved"]
coverage = (thefts.groupby("year")["coord_source"].value_counts(normalize=True).unstack(fill_value=0)
            .reindex(index=years, columns=coord_source_order, fill_value=0))

fig, ax = plt.subplots(figsize=config.FIGURE_SIZE_REGULAR)
coverage.plot(kind="bar", stacked=True, ax=ax, width=0.8,
              color=["#2166ac", "#67a9cf", "#d1e5f0", "#ef8a62", "#b2182b"])
ax.set_title("Coordinate Source by Year (vehicle theft reports)", pad=12)
ax.set_ylabel("Share of reports")
ax.set_xlabel("Year")
ax.set_xticklabels([str(y) for y in years], rotation=45, ha="right")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.24), ncol=5, fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "fig2_coord_coverage_by_year.png"), dpi=config.FIGURE_DPI, bbox_inches="tight")
plt.show()
print((coverage * 100).round(1).to_string())

#### Geocoding accuracy (validation)

Some reports appear twice in the raw files, once with stored LAPD coordinates and once without; the copy
without coordinates was geocoded independently. When both copies carry the **same street address**, the
distance between the stored point and the geocoded point measures geocoding error. (Copies with different
addresses under one DR are different places and are not comparable; they are reported separately.)
Results are split by the Census match type (Exact / Non_Exact, from the geocoding checkpoint keyed by CSV row
position) and by address type: numbered street address vs intersection (street + cross street, no number).
Table 12b gives the address-type mix of all geocoded theft points, i.e. how much of it the validation covers.

In [ ]:
cols = ["DR_NBR", "CRME_CLSS_PRMRY_CD", "coord_source", "lat_final", "lon_final", "STR_NBR", "STR_NM", "CRS_STR_NM", "data_year"]
pts = pd.read_csv(config.PREPROCESSED, usecols=cols, dtype=str)
pts["row"] = np.arange(len(pts)).astype(str)
census_ckpt = pd.read_csv(os.path.join(config.DATA_DIR, "phase3_census.csv"), dtype=str).drop_duplicates("orig_idx", keep="last")
pts = pts.merge(census_ckpt[["orig_idx", "match_type"]], left_on="row", right_on="orig_idx", how="left")
pts["lat_final"] = pd.to_numeric(pts["lat_final"], errors="coerce")
pts["lon_final"] = pd.to_numeric(pts["lon_final"], errors="coerce")
pts = pts.dropna(subset=["lat_final"])
for c in ["STR_NBR", "STR_NM", "CRS_STR_NM"]:
    pts[c] = pts[c].fillna("").str.strip().str.upper()
pts["address_type"] = np.where(pts["STR_NBR"] != "", "numbered", np.where(pts["CRS_STR_NM"] != "", "intersection", "street only"))
key = ["DR_NBR", "CRME_CLSS_PRMRY_CD"]
stored = pts[pts["coord_source"] == "stored"].drop_duplicates(key)
geo = pts[pts["coord_source"].isin(["census", "nominatim"])]
pairs = geo.merge(stored, on=key, suffixes=("_g", "_s"))
pairs["same_address"] = ((pairs["STR_NBR_g"] == pairs["STR_NBR_s"]) & (pairs["STR_NM_g"] == pairs["STR_NM_s"])
                         & (pairs["CRS_STR_NM_g"] == pairs["CRS_STR_NM_s"]) & (pairs["STR_NM_g"] != ""))
pairs["group"] = (np.where(pairs["coord_source_g"] == "census", "census " + pairs["match_type_g"].fillna("?"), pairs["coord_source_g"])
                  + ", " + pairs["address_type_g"])
pairs.loc[~pairs["same_address"], "group"] = "different address (not comparable)"


def haversine_m(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6_371_000 * 2 * np.arcsin(np.sqrt(a))


pairs["err_m"] = haversine_m(pairs["lat_final_g"], pairs["lon_final_g"], pairs["lat_final_s"], pairs["lon_final_s"])
geo_err = (pairs.groupby("group")["err_m"]
           .agg(n="size", median="median", p75=lambda s: s.quantile(.75), p90=lambda s: s.quantile(.9),
                within_100m=lambda s: (s <= 100).mean() * 100, within_500m=lambda s: (s <= 500).mean() * 100,
                over_2km=lambda s: (s > 2000).mean() * 100)
           .round(1))
geo_err.to_csv(os.path.join(config.TABLES_DIR, "table12_geocoding_error.csv"))

# Address-type mix of geocoded theft (510) points, by coordinate period
g510 = pts[(pts["CRME_CLSS_PRMRY_CD"].str.strip() == "510") & pts["coord_source"].isin(["census", "nominatim"])].copy()
g510["period"] = np.where(pd.to_numeric(g510["data_year"], errors="coerce").isin(config.GEOCODED_REGIME_YEARS),
                          "geocoded-location years", "stored-location years")
mix = (g510.groupby(["period", "coord_source", "address_type"]).size().rename("points").reset_index())
mix["share_of_geocoded_in_period_%"] = mix["points"] / mix.groupby("period")["points"].transform("sum") * 100
# Validated pairs (same report, same address, stored copy available), counted within the same period
val = pairs[pairs["same_address"]].copy()
val["period"] = np.where(pd.to_numeric(val["data_year_g"], errors="coerce").isin(config.GEOCODED_REGIME_YEARS),
                         "geocoded-location years", "stored-location years")
keys = list(zip(mix["period"], mix["coord_source"], mix["address_type"]))
v_all = val.groupby(["period", "coord_source_g", "address_type_g"]).size()
v_510 = val[val["CRME_CLSS_PRMRY_CD"].str.strip() == "510"].groupby(["period", "coord_source_g", "address_type_g"]).size()
# `points` counts geocoded theft (510) reports; validated pairs are reported for thefts and for all crime classes
mix["validated_pairs_theft_510"] = [int(v_510.get(k, 0)) for k in keys]
mix["validated_pairs_all_classes"] = [int(v_all.get(k, 0)) for k in keys]
mix.round(1).to_csv(os.path.join(config.TABLES_DIR, "table12b_geocoded_address_types.csv"), index=False)
print(mix.round(1).to_string(index=False))
print("Table 12 - Geocoded vs stored position for the same report (metres; % columns are shares)")
print(geo_err.to_string())

fig, ax = plt.subplots(figsize=config.FIGURE_SIZE_REGULAR)
bins = np.logspace(0, 5, 60)
for s, col in [("census Exact, numbered", "#2166ac"), ("census Non_Exact, numbered", "#67a9cf"), ("nominatim, numbered", "#b2182b")]:
    e = pairs.loc[pairs["group"] == s, "err_m"].clip(lower=1)
    ax.hist(e, bins=bins, color=col, alpha=0.55, label=f"{s} (n={len(e):,}, median {e.median():.0f} m)")
ax.axvline(config.KDE_BANDWIDTH_M, color="#555", ls="--", lw=1, label=f"KDE bandwidth ({config.KDE_BANDWIDTH_M} m)")
ax.set_xscale("log")
ax.set_xlabel("Distance between geocoded and stored location (m)")
ax.set_ylabel("Reports")
ax.set_title("Geocoding Error, Numbered Addresses (same report and address, stored vs geocoded copy)", pad=12)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "fig37_geocoding_error.png"), dpi=config.FIGURE_DPI, bbox_inches="tight")
plt.show()

#### Are reports without usable coordinates spatially random?

Every report carries an LAPD reporting district (RPT_DIST_NBR); its first digits give the LAPD geographic area
(RD // 100, 21 areas). If the share of reports without coordinates differs strongly between areas, excluding
them biases the maps towards well-geocoded areas.

In [ ]:
rd = pd.to_numeric(thefts["RPT_DIST_NBR"], errors="coerce")
thefts["lapd_area"] = (rd // 100).where(rd.between(100, 2199)).astype("Int64")
area_q = (thefts.dropna(subset=["lapd_area"]).groupby("lapd_area")
          .agg(reports=("DR_NBR", "size"), no_coords_pct=("lat_final", lambda s: s.isna().mean() * 100)))
area_q["share_of_all_thefts_pct"] = area_q["reports"] / area_q["reports"].sum() * 100
thefts["regime"] = np.where(thefts["year"].isin(config.GEOCODED_REGIME_YEARS), "geocoded_years", "stored_years")
by_regime = (thefts.dropna(subset=["lapd_area"])
             .groupby(["lapd_area", "regime"])["lat_final"].apply(lambda s: s.isna().mean() * 100).unstack())
area_q = area_q.join(by_regime.add_prefix("no_coords_pct_"))
area_q.to_csv(os.path.join(config.TABLES_DIR, "table18_missing_coords_by_area.csv"))
print(area_q.round(1).to_string())
from scipy.stats import chi2_contingency, spearmanr
tab = thefts.dropna(subset=["lapd_area"]).groupby("lapd_area")["lat_final"].agg(lambda s: [s.notna().sum(), s.isna().sum()])
chi2, p_area, _, _ = chi2_contingency(np.vstack(tab.values))
rho_area, p_rho = spearmanr(area_q["share_of_all_thefts_pct"], area_q["no_coords_pct"])
print(f"Missing-coordinate share ranges {area_q['no_coords_pct'].min():.1f}%-{area_q['no_coords_pct'].max():.1f}% across "
      f"{len(area_q)} LAPD areas (chi2 p = {p_area:.2g}); correlation with area theft volume: rho = {rho_area:.2f} (p = {p_rho:.2f})")

#### Occurrence-time quality by year

2022 records carry no occurrence time ('01-JAN-00'); '00:01' is LAPD's placeholder for an unknown time.
These are excluded from every hour-of-day analysis.

In [ ]:
time_q = thefts.groupby("year").agg(reports=("DR_NBR", "size"), time_unknown_pct=("time_unknown", "mean"))
time_q["time_unknown_pct"] = (time_q["time_unknown_pct"] * 100).round(1)
time_q.to_csv(os.path.join(config.TABLES_DIR, "table13_time_quality.csv"))
print(time_q.to_string())

#### Vehicle Recovery Rate by Year

Recovery rate = share of 510 reports for which a 521 (recovered) record with the same DR exists.
Recent years are right-censored: vehicles stolen late in 2022 had little time to be recovered before the
data were extracted.

In [ ]:
recovery_by_year = thefts.groupby("year").agg(total=("DR_NBR", "size"), recovered=("recovered", "sum"),
                                              median_days=("days_to_recovery", "median"))
recovery_by_year["recovery_rate"] = recovery_by_year["recovered"] / recovery_by_year["total"] * 100

fig, ax = plt.subplots(figsize=config.FIGURE_SIZE_REGULAR)
ax.plot(recovery_by_year.index, recovery_by_year["recovery_rate"], marker="o", linewidth=2.5, color="#2166ac")
ax.set_title("Share of Stolen Vehicles Later Recovered (510 reports with a linked 521 record)")
ax.set_xlabel("Year of theft")
ax.set_ylabel("Recovered (%)")
ax.set_ylim(0, 100)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "fig3_recovery_rate.png"), dpi=config.FIGURE_DPI, bbox_inches="tight")
plt.show()
print(recovery_by_year.round(1).to_string())